# 🌴 Taller LiDAR Forestal · Cuaderno 2
## De la ortofoto a los árboles con *Deep Learning*

**Kaab Map · Sesión final del taller (≈ 45 min)**

En el Cuaderno 1 contamos árboles con el **CHM** del LiDAR. Pero un vuelo LiDAR es caro; una **ortofoto RGB** (dron o satélite) es mucho más barata y común. ¿Podemos detectar árboles **solo con la imagen**?

Para eso usaremos **aprendizaje profundo** (*Deep Learning*): modelos que **aprenden a partir de ejemplos** en lugar de seguir reglas que escribimos a mano.

| Parte | Tema | Modelo |
|---|---|---|
| 1 | Una imagen es una matriz de números · índices de verdor | Regla manual (ExG) |
| 2 | El LiDAR como "maestro": etiquetas automáticas | — |
| 3 | Clasificación de parches: ¿árbol o no árbol? | Red neuronal convolucional (CNN) |
| 4 | Detección de objetos: encontrar y contar cada palma | YOLO |
| 5 | Comparación de métodos | — |

> ⚡ **Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**. La parte 4 (YOLO) entrena mucho más rápido con GPU.

In [ ]:
#@title ⚙️ Paso 0 · Preparar el entorno y descargar los datos (ejecuta y espera ~1 min)
#@markdown Si el instructor te compartió un **ID de Google Drive** del archivo `datos_taller.zip`, pégalo aquí.
#@markdown Si lo dejas vacío, se te pedirá subir el ZIP desde tu computadora.
ID_ZIP_DRIVE = ""  #@param {type:"string"}

import os, sys, subprocess, zipfile, warnings
warnings.filterwarnings("ignore")
EN_COLAB = "google.colab" in sys.modules

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import rasterio
except ImportError:
    _pip("rasterio")

if not os.path.exists("datos/zona1_plantacion_chm.tif"):
    if ID_ZIP_DRIVE.strip():
        _pip("gdown")
        import gdown
        gdown.download(id=ID_ZIP_DRIVE.strip(), output="datos_taller.zip", quiet=False)
    elif EN_COLAB:
        from google.colab import files
        print("Selecciona el archivo datos_taller.zip ...")
        subido = files.upload()
        os.rename(list(subido.keys())[0], "datos_taller.zip")
    with zipfile.ZipFile("datos_taller.zip") as z:
        z.extractall(".")

print("Archivos disponibles en la carpeta datos/:")
for f in sorted(os.listdir("datos")):
    print(f"  {f:32s} {os.path.getsize('datos/' + f) / 1e6:5.1f} MB")

In [ ]:
#@title 🧰 Funciones de apoyo (no necesitas leerlas, solo ejecútala)
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from skimage.filters import gaussian
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
from scipy.spatial import cKDTree
from scipy import ndimage as ndi
plt.rcParams.update({"figure.dpi": 100, "font.size": 10, "axes.titlesize": 11})


def cargar_zona(nombre):
    with rasterio.open(f"datos/{nombre}_chm.tif") as src:
        chm = src.read(1).astype("float32"); tr, crs = src.transform, src.crs
    with rasterio.open(f"datos/{nombre}_orto.tif") as src:
        rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
    res = tr.a
    return dict(nombre=nombre, chm=chm, rgb=rgb, transform=tr, crs=crs, res=res,
                ext=[0, chm.shape[1] * res, chm.shape[0] * res, 0])


def lidar_referencia(zona, sigma_m=0.4, dist_m=3.0, hmin=2.0, frac=0.6):
    """Watershed del Cuaderno 1 + máscara de 'copa': la parte de cada árbol
    por arriba del 60 % de su altura (evita las mesetas del CHM entre copas)."""
    res = zona["res"]
    suave = gaussian(zona["chm"], sigma=sigma_m / res, preserve_range=True)
    cimas = peak_local_max(suave, min_distance=int(dist_m / res), threshold_abs=hmin, exclude_border=False)
    mk = np.zeros(suave.shape, np.int32); mk[cimas[:, 0], cimas[:, 1]] = np.arange(1, len(cimas) + 1)
    copas = watershed(-suave, mk, mask=suave > hmin)
    hmax = np.r_[0, ndi.maximum(suave, copas, np.arange(1, copas.max() + 1))]
    mascara = (copas > 0) & (suave >= frac * hmax[copas])
    zona.update(mascara=mascara, cimas=cimas, copas=copas, suave=suave)
    return zona


def ventana(ax, x0, y0, lado):
    ax.set_xlim(x0, x0 + lado); ax.set_ylim(y0 + lado, y0)


def crear_parches(zona, n, P, semilla=0):
    """Toma n parches de P×P px de la ortofoto, balanceados 50 % árbol / 50 % no árbol.
    La etiqueta la pone el LiDAR: 1 si el centro del parche está sobre la parte alta de una copa."""
    rng = np.random.default_rng(semilla)
    m, rgb = zona["mascara"], zona["rgb"]
    h, w = m.shape; r = P // 2
    filas, cols = np.mgrid[r:h - r:4, r:w - r:4]
    filas, cols = filas.ravel(), cols.ravel()
    etiqueta = m[filas, cols]
    X, y = [], []
    for clase in (1, 0):
        idx = rng.choice(np.where(etiqueta == clase)[0], n // 2, replace=False)
        for i in idx:
            X.append(rgb[filas[i] - r:filas[i] + r, cols[i] - r:cols[i] + r]); y.append(clase)
    X, y = np.array(X), np.array(y)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


def mapa_probabilidad(modelo, zona, P, paso):
    """Desliza la red sobre toda la ortofoto y devuelve un mapa de probabilidad de 'árbol'."""
    rgb = zona["rgb"]; h, w, _ = rgb.shape; r = P // 2
    filas = np.arange(r, h - r, paso); cols = np.arange(r, w - r, paso)
    parches = np.array([rgb[f - r:f + r, c - r:c + r] for f in filas for c in cols])
    prob = modelo.predict(parches, batch_size=512, verbose=0).reshape(len(filas), len(cols))
    ref = zona["mascara"][np.ix_(filas, cols)]
    ext = [(cols[0] - paso / 2) * zona["res"], (cols[-1] + paso / 2) * zona["res"],
           (filas[-1] + paso / 2) * zona["res"], (filas[0] - paso / 2) * zona["res"]]
    return prob, ref, ext


def emparejar(pred_xy, ref_xy, dist_max):
    """Empareja detecciones con árboles de referencia (uno a uno, el más cercano primero)."""
    if len(pred_xy) == 0:
        return np.zeros(0, bool), np.zeros(len(ref_xy), bool)
    d, j = cKDTree(ref_xy).query(pred_xy, distance_upper_bound=dist_max)
    tp_pred = np.zeros(len(pred_xy), bool); tp_ref = np.zeros(len(ref_xy), bool)
    for i in np.argsort(d):
        if np.isfinite(d[i]) and not tp_ref[j[i]]:
            tp_pred[i] = tp_ref[j[i]] = True
    return tp_pred, tp_ref

print("Funciones listas ✔")

---
## Parte 1 · Una imagen es una matriz de números

Para la computadora, la ortofoto es una **matriz de 1500 × 1500 píxeles × 3 bandas** (Rojo, Verde, Azul). Cada valor va de 0 (oscuro) a 255 (brillante). Con 10 cm por píxel, 1500 px = 150 m.

In [ ]:
z1 = lidar_referencia(cargar_zona("zona1_plantacion"))
rgb = z1["rgb"]
print("Forma de la matriz:", rgb.shape, "→ (filas, columnas, bandas)")
print("Valores de un recorte de 4 × 4 píxeles en la banda VERDE:")
print(rgb[800:804, 1000:1004, 1])

In [ ]:
#@title 🖼️ Las tres bandas por separado
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(rgb, extent=z1["ext"]); ax[0].set_title("Composición RGB")
for i, (nom, cm) in enumerate([("Rojo", "Reds_r"), ("Verde", "Greens_r"), ("Azul", "Blues_r")]):
    ax[i + 1].imshow(rgb[..., i], cmap=cm, extent=z1["ext"]); ax[i + 1].set_title(f"Banda {nom}")
for a in ax: ventana(a, 60, 60, 30)
plt.tight_layout(); plt.show()

### Una regla escrita a mano: el índice de verdor ExG

Antes del *Deep Learning*, la forma clásica era diseñar **reglas**. Por ejemplo, el *Excess Green Index* (Woebbecke et al., 1995):

$$\text{ExG} = 2G - R - B$$

Si `ExG > umbral` → "vegetación". Comparémoslo con lo que dice el LiDAR: la **parte alta de cada copa** (píxeles por arriba del 60 % de la altura de su árbol, según el watershed del Cuaderno 1).

✏️ **Ejercicio 1:** mueve el umbral. ¿Puedes lograr que el ExG coincida con el LiDAR? Después cambia a `zona2_natural`. ¿Qué pasa?

In [ ]:
#@title 🌿 ExG vs. LiDAR
UMBRAL_EXG = 20  #@param {type:"slider", min:-20, max:80, step:5}
ZONA_EXG = "zona1_plantacion"  #@param ["zona1_plantacion", "zona2_natural"]
ze = z1 if ZONA_EXG == "zona1_plantacion" else lidar_referencia(cargar_zona(ZONA_EXG))
f = ze["rgb"].astype("float32")
exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
veg_exg = gaussian(exg, 2) > UMBRAL_EXG
acuerdo = (veg_exg == ze["mascara"]).mean() * 100

fig, ax = plt.subplots(1, 3, figsize=(20, 6.5))
ax[0].imshow(ze["rgb"], extent=ze["ext"]); ax[0].set_title(f"Ortofoto · {ZONA_EXG}")
ax[1].imshow(veg_exg, cmap="Greens", extent=ze["ext"]); ax[1].set_title(f"ExG > {UMBRAL_EXG} → 'vegetación'")
ax[2].imshow(ze["mascara"], cmap="Greens", extent=ze["ext"]); ax[2].set_title("Referencia LiDAR: parte alta de cada copa")
for a in ax: ventana(a, 40, 40, 60)
plt.tight_layout(); plt.show()
print(f"Coincidencia píxel a píxel ExG vs LiDAR: {acuerdo:.1f} %")

🔎 **Pregunta:** en la plantación el ExG funciona razonablemente (palmas verdes sobre arena), pero en la vegetación natural **todo es verde**: no distingue **pasto o arbusto** de **copa de árbol**, ni separa una copa de otra. Además, las **sombras** y frondas secas no son "verdes". Un píxel aislado no tiene suficiente información: hay que mirar su **contexto** (forma, textura, vecinos). Eso es justo lo que hace una **red convolucional**.

---
## Parte 2 · El LiDAR como "maestro"

Una red neuronal aprende con **aprendizaje supervisado**: le mostramos miles de ejemplos **con su respuesta correcta** (etiqueta) y ajusta sus parámetros hasta equivocarse lo menos posible.

El problema típico es que **etiquetar a mano es lentísimo**. Aquí tenemos un truco: el **CHM del LiDAR ya sabe dónde hay árboles**, así que lo usamos para etiquetar automáticamente la ortofoto:

* Recortamos **parches** de 32 × 32 px (3.2 × 3.2 m) de la ortofoto.
* Si el centro del parche cae en la **parte alta de una copa** según el CHM → etiqueta **1 = árbol**; si no → **0 = no árbol**.

Así, el modelo entrenado podrá usarse después **en zonas donde solo hay ortofoto**.

In [ ]:
P = 32                                   # tamaño del parche en píxeles (3.2 m)
X, y = crear_parches(z1, n=6000, P=P)    # 6000 ejemplos de la zona 1, etiquetados por el LiDAR
print("Parches:", X.shape, "→ (ejemplos, alto, ancho, bandas)")
print("Árbol:", (y == 1).sum(), " · No árbol:", (y == 0).sum())

In [ ]:
#@title 🖼️ Algunos ejemplos de entrenamiento
fig, ax = plt.subplots(3, 8, figsize=(16, 6.5))
for i, a in enumerate(ax.ravel()):
    a.imshow(X[i]); a.set_title("🌴 árbol" if y[i] else "suelo / otro", color="green" if y[i] else "saddlebrown", fontsize=9)
    a.axis("off")
plt.suptitle("Parches de 3.2 × 3.2 m etiquetados automáticamente con el CHM", y=1.0)
plt.tight_layout(); plt.show()

---
## Parte 3 · Una red neuronal convolucional (CNN) básica

Una **CNN** aprende **filtros** (pequeñas ventanas de 3 × 3 píxeles) que detectan patrones: primero bordes y colores, luego texturas (frondas, sombras), y al final formas completas.

```
Parche RGB 32×32×3
   │  Conv 3×3 (16 filtros) + ReLU   → detecta bordes / colores
   │  MaxPooling 2×2                 → reduce a 16×16 (se queda con lo más fuerte)
   │  Conv 3×3 (32 filtros) + ReLU   → detecta texturas
   │  MaxPooling 2×2                 → 8×8
   │  Conv 3×3 (64 filtros) + ReLU   → detecta partes de copa
   │  GlobalAveragePooling           → un número por filtro
   │  Dense (32) + ReLU
   ▼  Dense (1) + Sigmoide           → probabilidad de "árbol" (0 a 1)
```

* **Entrenar** = ajustar los pesos de los filtros para minimizar el error (*función de pérdida*), con el algoritmo de **descenso del gradiente** (aquí, la variante *Adam*).
* Una **época** = la red vio **todos** los ejemplos una vez.
* Separamos un **20 % de validación**: ejemplos que la red **no usa para aprender**, solo para vigilar que no esté **memorizando** (*sobreajuste*).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

red = keras.Sequential([
    keras.Input(shape=(P, P, 3)),
    layers.Rescaling(1 / 255),                                   # 0–255 → 0–1
    layers.Conv2D(16, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),                       # probabilidad de árbol
])
red.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
red.summary()

🔎 Fíjate en el **número total de parámetros** del resumen: son los "pesos" que la red debe aprender. ¿Es mucho o poco comparado con los 6000 ejemplos?

✏️ **Ejercicio 2:** entrena la red. Mueve `EPOCAS` (prueba 2 y 15) y observa las curvas.

In [ ]:
#@title 🏋️ Entrenar la red
EPOCAS = 8  #@param {type:"slider", min:1, max:20, step:1}
keras.utils.set_random_seed(42)
red = keras.models.clone_model(red); red.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
historia = red.fit(X, y, epochs=EPOCAS, batch_size=64, validation_split=0.2, verbose=2)

h = historia.history
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(h["loss"], "o-", label="entrenamiento"); ax[0].plot(h["val_loss"], "s-", label="validación")
ax[0].set_title("Pérdida (error) — debe bajar"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(h["accuracy"], "o-", label="entrenamiento"); ax[1].plot(h["val_accuracy"], "s-", label="validación")
ax[1].set_title("Exactitud — debe subir"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Exactitud en validación: {100 * h['val_accuracy'][-1]:.1f} %")

In [ ]:
#@title 🔬 ¿Qué "ve" la red? Mapas de activación de la primera capa
ejemplo = X[np.where(y == 1)[0][0]]
capa1 = keras.Model(red.inputs, red.layers[1].output)
act = capa1.predict(ejemplo[None], verbose=0)[0]
fig, ax = plt.subplots(2, 9, figsize=(18, 5.2), gridspec_kw={"hspace": 0.35})
ax[0, 0].imshow(ejemplo); ax[0, 0].set_title("entrada"); ax[1, 0].axis("off")
huecos = [a for a in ax.ravel() if a not in (ax[0, 0], ax[1, 0])]
for k, a in enumerate(huecos):
    a.imshow(act[..., k], cmap="magma"); a.set_title(f"filtro {k + 1}", fontsize=8)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.suptitle("Cada filtro aprendió a resaltar algo distinto: bordes, verdes, sombras…", y=1.02)
plt.tight_layout(); plt.show()

### Prueba en una zona que la red **nunca vio**

Ahora deslizamos la red sobre toda la ortofoto de otra zona, parche por parche, y obtenemos un **mapa de probabilidad de árbol**. Lo comparamos con el LiDAR de esa zona (que la red no conoce).

✏️ **Ejercicio 3:** primero prueba en `zona3_prueba` (otra plantación). Después cambia a `zona2_natural` (vegetación natural). ¿Qué pasa con la exactitud? ¿Por qué?

In [ ]:
#@title 🗺️ Mapa de probabilidad en una zona nueva
ZONA_PRUEBA = "zona3_prueba"  #@param ["zona3_prueba", "zona2_natural"]
UMBRAL = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
zp = lidar_referencia(cargar_zona(ZONA_PRUEBA))
prob, ref, ext_p = mapa_probabilidad(red, zp, P, paso=8)
pred = prob > UMBRAL

vp = (pred & ref).sum(); fp = (pred & ~ref).sum(); fn = (~pred & ref).sum(); vn = (~pred & ~ref).sum()
exactitud = (vp + vn) / ref.size; iou = vp / (vp + fp + fn)
errores = np.zeros(pred.shape + (3,)); errores[pred & ref] = (0.2, 0.7, 0.2)
errores[pred & ~ref] = (0.9, 0.2, 0.2); errores[~pred & ref] = (0.2, 0.4, 0.9); errores[~pred & ~ref] = (1, 1, 1)

fig, ax = plt.subplots(1, 4, figsize=(22, 6))
ax[0].imshow(zp["rgb"], extent=zp["ext"]); ax[0].set_title(f"Ortofoto · {ZONA_PRUEBA}")
im = ax[1].imshow(prob, cmap="YlGn", vmin=0, vmax=1, extent=ext_p); ax[1].set_title("CNN: probabilidad de árbol")
plt.colorbar(im, ax=ax[1], shrink=.7)
ax[2].imshow(ref, cmap="Greens", extent=ext_p); ax[2].set_title("Referencia LiDAR (parte alta de copas)")
ax[3].imshow(errores, extent=ext_p); ax[3].set_title("Verde = acierto · Rojo = falso árbol · Azul = árbol omitido")
plt.tight_layout(); plt.show()
print(f"Exactitud global: {100 * exactitud:.1f} %   ·   IoU de la clase árbol: {100 * iou:.1f} %")
display(pd.DataFrame([[vp, fn], [fp, vn]], index=["LiDAR: árbol", "LiDAR: no árbol"],
                     columns=["CNN: árbol", "CNN: no árbol"]).rename_axis("Matriz de confusión"))

🔎 **Discusión:**
* La CNN produce un **mapa de cobertura** de copas, pero **no separa** un árbol de otro: no puede contarlos. Para eso necesitamos **detección de objetos**.
* Si la exactitud cae en la zona natural, es un problema de **generalización**: la red solo aprendió cómo se ven las palmas sobre arena. Un modelo es tan bueno como la **diversidad de sus datos de entrenamiento**.

---
## Parte 4 · Detección de objetos con YOLO

**YOLO** (*You Only Look Once*, Redmon et al., 2016) es una familia de redes que, en una sola pasada, predice para cada objeto:

* una **caja** (*bounding box*): centro x, centro y, ancho, alto;
* una **clase** (aquí solo una: `palma`);
* una **confianza** (0–1): qué tan segura está la red.

Usaremos **YOLO11-nano** de la librería `ultralytics`, **preentrenado** con millones de imágenes cotidianas (COCO). Lo vamos a **ajustar** (*fine-tuning* / *transfer learning*) con nuestras palmas: la red ya sabe "ver" bordes y formas, solo le enseñamos qué es una palma.

**¿De dónde salen las cajas de entrenamiento?** Otra vez del LiDAR: cada copa del watershed (Cuaderno 1) se convierte en una caja que encierra la parte de la copa por arriba del **60 % de su altura** (la misma regla de la Parte 2).

In [ ]:
#@title 📦 Instalar YOLO (ultralytics) y revisar la GPU
try:
    import ultralytics
except ImportError:
    _pip("ultralytics")
import ultralytics, torch
from ultralytics import YOLO
GPU = torch.cuda.is_available()
print("ultralytics", ultralytics.__version__, "· GPU disponible:", GPU)
if not GPU:
    print("⚠️ Sin GPU el entrenamiento será lento. Cambia el entorno a GPU T4 si puedes.")

In [ ]:
#@title 🏷️ Cajas de entrenamiento generadas con el LiDAR (zona 1)
def cajas_desde_copas(zona, frac=0.6):
    cajas = []
    for r in regionprops(zona["copas"], intensity_image=zona["suave"]):
        m = r.image & (r.image_intensity >= frac * r.intensity_max)
        ff, cc = np.nonzero(m)
        cajas.append((cc.min() + r.bbox[1], ff.min() + r.bbox[0], cc.max() + r.bbox[1] + 1, ff.max() + r.bbox[0] + 1))
    return np.array(cajas)          # (x0, y0, x1, y1) en píxeles

cajas_z1 = cajas_desde_copas(z1)
fig, ax = plt.subplots(1, 2, figsize=(18, 8))
for a, lado in zip(ax, [150, 40]):
    a.imshow(z1["rgb"], extent=z1["ext"])
    for x0, y0, x1, y1 in cajas_z1 * z1["res"]:
        a.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1.5 if lado < 100 else .6))
    ventana(a, 0 if lado == 150 else 60, 0 if lado == 150 else 60, lado)
ax[0].set_title(f"{len(cajas_z1)} cajas de entrenamiento (zona 1)"); ax[1].set_title("Zoom 40 × 40 m")
plt.tight_layout(); plt.show()

### Preparar el conjunto de datos en formato YOLO

YOLO necesita imágenes y, para cada una, un archivo `.txt` con una línea por objeto:

```
clase  x_centro  y_centro  ancho  alto      ← todo normalizado de 0 a 1
0      0.4312    0.2290    0.1180 0.1240
```

Dividimos la zona 1 **espacialmente**: de los 100 m del norte sacamos **64 recortes aleatorios de 50 × 50 m** (500 px) para **entrenar**, y los 50 m del sur (3 mosaicos que el modelo no ve al entrenar) sirven para **validar**. Separar por zonas, y no al azar, evita que la validación sea "trampa" (parches vecinos se parecen mucho: *autocorrelación espacial*).

In [ ]:
#@title 🗂️ Crear dataset YOLO
import os, shutil
from PIL import Image
T = 500
rng = np.random.default_rng(0)
if os.path.exists("dataset_yolo"): shutil.rmtree("dataset_yolo")

def guardar_mosaico(sub, nombre, f0, c0):
    os.makedirs(f"dataset_yolo/images/{sub}", exist_ok=True); os.makedirs(f"dataset_yolo/labels/{sub}", exist_ok=True)
    Image.fromarray(z1["rgb"][f0:f0 + T, c0:c0 + T]).save(f"dataset_yolo/images/{sub}/{nombre}.png")
    lineas = []
    for x0, y0, x1, y1 in cajas_z1:
        a0, b0, a1, b1 = max(x0, c0), max(y0, f0), min(x1, c0 + T), min(y1, f0 + T)
        if a1 <= a0 or b1 <= b0: continue
        if (a1 - a0) * (b1 - b0) < 0.4 * (x1 - x0) * (y1 - y0): continue   # muy cortada por el borde
        lineas.append(f"0 {((a0 + a1) / 2 - c0) / T:.5f} {((b0 + b1) / 2 - f0) / T:.5f} {(a1 - a0) / T:.5f} {(b1 - b0) / T:.5f}")
    open(f"dataset_yolo/labels/{sub}/{nombre}.txt", "w").write("\n".join(lineas))

for k in range(64):                                   # entrenamiento: 100 m del norte
    guardar_mosaico("train", f"t{k}", int(rng.integers(0, 1000 - T + 1)), int(rng.integers(0, 1500 - T + 1)))
for k, c0 in enumerate([0, 500, 1000]):               # validación: 50 m del sur
    guardar_mosaico("val", f"v{k}", 1000, c0)
with open("dataset_yolo/data.yaml", "w") as fh:
    fh.write(f"path: {os.path.abspath('dataset_yolo')}\ntrain: images/train\nval: images/val\nnames:\n  0: palma\n")
print(open("dataset_yolo/data.yaml").read())
print("Ejemplo de etiquetas:", open("dataset_yolo/labels/train/t0.txt").read().splitlines()[:3])

### Entrenar (≈ 2–3 min con GPU · ≈ 5 min sin GPU)

Algunos conceptos que verás en la salida del entrenamiento:

* **box_loss**: error en la posición y tamaño de las cajas · **cls_loss**: error de clase · **dfl_loss**: refinamiento de bordes.
* **Precision (P)**: de lo que detectó, ¿qué fracción eran palmas reales?
* **Recall (R)**: de las palmas reales, ¿qué fracción encontró?
* **mAP50**: precisión promedio considerando correcta una caja si se traslapa ≥ 50 % (**IoU** ≥ 0.5) con la real.

YOLO además hace **aumento de datos** (*data augmentation*): voltea, cambia brillo y combina mosaicos, para que pocas imágenes "parezcan" muchas más.

In [ ]:
EPOCAS_YOLO = 40 if GPU else 20      # con GPU ≈ 2–3 min

yolo = YOLO("yolo11n.pt")                                  # modelo preentrenado (nano)
resultado = yolo.train(data="dataset_yolo/data.yaml", epochs=EPOCAS_YOLO, imgsz=512,
                       batch=8, patience=0, plots=False, verbose=False, seed=0, workers=2,
                       project="runs", name="palmas", exist_ok=True)

### Detectar y contar palmas en la zona 3 (nunca vista)

Pasamos la ortofoto completa de la zona 3 por el modelo y comparamos sus detecciones con las **cimas del LiDAR** de esa zona. Una detección es correcta si su centro está a **≤ 3 m** de una cima (menos de la mitad del marco de 8 m; la cima del CHM no siempre coincide con el centro visual de la copa).

La zona 3 tiene dos partes: **plantación** (sur) y una franja de **matorral natural** al norte (y < 35 m), un tipo de vegetación que el modelo casi no vio al entrenar. Por eso reportamos las métricas de ambas.

* **VP** (verdadero positivo): detectó una palma real. **FP** (falso positivo): detectó algo que no es palma. **FN** (falso negativo): se le escapó una palma.
* $\text{Precisión} = \frac{VP}{VP+FP}$ · $\text{Exhaustividad (recall)} = \frac{VP}{VP+FN}$ · $F_1 = \frac{2PR}{P+R}$

✏️ **Ejercicio 4:** cambia `CONFIANZA_MIN`. ¿Qué pasa con la precisión y la exhaustividad? ¿Qué valor usarías para un inventario?

In [ ]:
#@title 🎯 Detección en la zona 3 y evaluación contra el LiDAR
CONFIANZA_MIN = 0.25  #@param {type:"slider", min:0.05, max:0.9, step:0.05}
z3 = lidar_referencia(cargar_zona("zona3_prueba"))
det = yolo.predict(z3["rgb"][..., ::-1], imgsz=1504, conf=CONFIANZA_MIN, iou=0.3, max_det=2000, verbose=False)[0]
cajas = det.boxes.xyxy.cpu().numpy(); conf = det.boxes.conf.cpu().numpy()
centros = np.c_[(cajas[:, 0] + cajas[:, 2]) / 2, (cajas[:, 1] + cajas[:, 3]) / 2] * z3["res"]
ref_xy = z3["cimas"][:, ::-1] * z3["res"]
ok_pred, ok_ref = emparejar(centros, ref_xy, dist_max=3.0)

def resumen(sel_pred, sel_ref):
    vp, fp, fn = int((ok_pred & sel_pred).sum()), int((~ok_pred & sel_pred).sum()), int((~ok_ref & sel_ref).sum())
    p, r = vp / max(vp + fp, 1), vp / max(vp + fn, 1)
    return {"Árboles LiDAR (referencia)": int(sel_ref.sum()), "Detecciones YOLO": int(sel_pred.sum()), "VP": vp, "FP": fp, "FN": fn,
            "Precisión": round(p, 3), "Exhaustividad (recall)": round(r, 3), "F1": round(2 * p * r / max(p + r, 1e-9), 3)}

fig, ax = plt.subplots(1, 2, figsize=(20, 9.5))
for a in ax:
    a.imshow(z3["rgb"], extent=z3["ext"])
    for (x0, y0, x1, y1), ok in zip(cajas * z3["res"], ok_pred):
        a.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="lime" if ok else "red", lw=1.2))
    a.plot(ref_xy[~ok_ref, 0], ref_xy[~ok_ref, 1], "o", mfc="none", mec="deepskyblue", ms=14, mew=2.5)
ax[0].axhline(35, color="white", ls="--", lw=1.5)
ax[0].set_title(f"YOLO: {len(cajas)} detecciones · verde = VP, rojo = FP, azul = FN")
ax[1].set_title("Zoom 40 × 40 m"); ventana(ax[1], 60, 60, 40)
plt.tight_layout(); plt.show()

display(pd.DataFrame({"Zona 3 completa": resumen(np.ones(len(centros), bool), np.ones(len(ref_xy), bool)),
                      "Solo plantación (y > 35 m)": resumen(centros[:, 1] > 35, ref_xy[:, 1] > 35)}, dtype=object))

🔎 **Preguntas:**
* ¿Dónde se equivoca el modelo? Compara la plantación con el matorral del norte. (Pista: cambio de dominio; también bordes de la imagen y palmas jóvenes).
* La "verdad" que usamos también viene de un algoritmo (watershed). ¿Qué implica eso? → Los errores del maestro **se heredan**. En un proyecto real la referencia debe ser **conteo en campo o fotointerpretación revisada**.
* Con más zonas de entrenamiento (otras plantaciones, otras fechas, otras especies), ¿qué esperarías?

In [ ]:
#@title 💾 Exportar detecciones a GeoPackage (para QGIS)
tr = z3["transform"]
geo = [box(tr.c + x0 * tr.a, tr.f + y1 * tr.e, tr.c + x1 * tr.a, tr.f + y0 * tr.e) for x0, y0, x1, y1 in cajas]
gdf = gpd.GeoDataFrame({"confianza": conf.round(3), "valida_lidar": ok_pred}, geometry=geo, crs=z3["crs"])
gdf.to_file("detecciones_yolo_zona3.gpkg", layer="cajas", driver="GPKG")
pts = gdf.copy(); pts["geometry"] = gdf.centroid
pts.to_file("detecciones_yolo_zona3.gpkg", layer="centros", driver="GPKG")
print(f"detecciones_yolo_zona3.gpkg → {len(gdf)} cajas y centros")
if EN_COLAB:
    from google.colab import files
    files.download("detecciones_yolo_zona3.gpkg")

---
## Parte 5 · ¿Qué método usar?

| | **Watershed sobre CHM** | **CNN de parches** | **YOLO (detección)** |
|---|---|---|---|
| Insumo | LiDAR (CHM) | Ortofoto RGB | Ortofoto RGB |
| Necesita entrenamiento | No (solo calibrar parámetros) | Sí (miles de parches etiquetados) | Sí (cientos de cajas) |
| Resultado | Cimas + polígonos de copa + **altura** | Mapa de cobertura | Cajas + conteo |
| Cuenta árboles | ✅ | ❌ | ✅ |
| Mide altura | ✅ | ❌ | ❌ |
| Punto fuerte | Estructura 3D, sin etiquetas | Simple, rápido | Muy bueno en objetos repetitivos (plantaciones) |
| Punto débil | Copas entrelazadas | No separa individuos | Necesita ejemplos de cada tipo de vegetación |

**La idea clave del taller:** LiDAR e imágenes **se complementan**. El LiDAR da etiquetas y alturas precisas en una zona pequeña; el *Deep Learning* permite **escalar** a grandes superficies con imágenes más baratas.

✏️ **Reto final (si sobra tiempo):** en el Cuaderno 1 obtuviste la altura de cada palma. ¿Cómo combinarías las detecciones de YOLO con un CHM para construir un inventario completo? Discútelo con tu equipo.